## Scraping lyrics from Genius

Genius.com has a web API service for retrieving song URLs by artist's name. 
You can start by setting up a token to let you access their API and pull the song URLs. 
* Here's [documentation for the Genius API](https://docs.genius.com/#/getting-started-h1).
* And here's [info on the access token you need](https://docs.genius.com/#authentication).

But usually we want to retrieve lyrics based on albums (e.g. compare early vs. late albums by an artist). This is trickier to retrieve: we've never had much luck with this directly from Genius, but it helps to work with a combination of Genius and BeautifulSoup. Also, when we start pulling songs with Genius's API we've typically pulled in ads and had to try to clean them out. We want to scrape more cleanly and dodge the ads, so that is what we're attempting in this notebook.


### Working with API info: never push your API details to GitHub!

When you set up your API client, you're going to need [to create (or log in with) a Genius.com account](https://genius.com/signup), and once you're logged in it will send you here to [the API Clients page](https://genius.com/api-clients). 
To set up an API token Genius is going to require that you give them an "App URL": just give them one of your websites, and where it asks for a logo, give them a link to an image on your site. (It's looking for a favicon, and I have one stored at the root of my newtfire website at favicon.ico. So that is what I used.)  You'll receive a client secret and an access token. Copy and store them somewhere secret and safe.

Your Python script is going to need to access these things, but you must NOT push your secret API access info to GitHub! 

**So, here's what we do to secure your Python scraping script:**
* You create a system "dot" file that must be named `.env`
* Save it with your Python script or somewhere in your repo. (It's easiest for you if it's with your Python script.)
* Edit your `.gitignore` file (or set this up if you don't have it!): Add `.env` to it (or make sure it is already there). Storing that string in your `.gitignore` prevents it from being pushed to GitHub.
    * Double check that `.env` is properly screened out after you save it and go to `git add` your directory with the Python and `.env` file.
      NOTE: You should NOT see the `.env` being staged by git and should NEVER be in a git commit. We discovered that our textAnalysis-Hub .gitignore had been screening `.env/` as a directory and not the file, so we needed to add a line to our `.gitignore` while writing this notebook! 
* Your Python script will need to read the secret system dot `.env` file, and there's actually a handy Python library for that called `python-dotenv`.
     * Navigate to your directory that holds your .venv and activate it
     * `pip install python-dotenv`



In [1]:
import asyncio
import nodriver as uc
import requests
import time
import os
import re  # this is for regex substitutions! 
from dotenv import load_dotenv, find_dotenv
from bs4 import BeautifulSoup

load_dotenv(find_dotenv())

GENIUS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN")
headers = {"Authorization": f"Bearer {GENIUS_TOKEN}"}

import sys
print(sys.executable)

# ebb: CAREFUL! Don't print this to output in your notebook and then push the notebook with all its return cells! 
# Always clear your notebook console by going to Kernel >> Restart Notebook and Clear Outputs of All Cells

/Users/eeb4/Documents/GitHub/newtfire/textAnalysis-Hub/Class-Examples/Python/webScrapers/lyrics/.venv/bin/python3.12


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

path = find_dotenv()
print("Found .env at:", repr(path))        # '' means it was NOT found
print("Loaded:", load_dotenv(path))        # True = loaded something
print("Token present:", os.getenv("GENIUS_ACCESS_TOKEN") is not None)
print("Working dir:", os.getcwd())

In [ ]:
# import requests
from curl_cffi import requests

# headers_page = {"User-Agent": "Mozilla/5.0", "Accept-Language": "en-US,en;q=0.9"}
headers_page = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9"}

try:
    r = requests.get("https://genius.com/albums/Paramore/All-we-know-is-falling",
                     headers=headers_page, timeout=15)
    print("Status:", r.status_code)
    print("Final URL:", r.url)
    print("Page length:", len(r.text))
except Exception as e:
    print("Request failed:", type(e).__name__, e)

In [ ]:
print(r.headers.get("server"), r.headers.get("cf-mitigated"))
print(r.text[:600])

In [ ]:
r = requests.get("https://api.genius.com/albums/25380", headers=headers, timeout=15)
print(r.status_code)
if r.status_code == 200:
    print(r.json()["response"]["album"]["name"])

### Scraping an album's lyrics

In [ ]:
## ebb: THIS CELL IS FOR TESTING TO SEE WHAT <div> ELEMENTS ARE INSIDE THE data-lyrics-container
## SKIP if you don't need it. :-) 
r = requests.get("https://genius.com/Paramore-whoa-lyrics",
                     headers=headers_page, timeout=15)
soup = BeautifulSoup(r.text, "html.parser")

lyric_divs = soup.find_all("div", attrs={"data-lyrics-container": "true"})
for i, div in enumerate(lyric_divs):
    print(f"--- div {i} ---")
    print(div.get_text()[:200])
    print()

In [2]:
async def get_lyrics():
    # Starts a stealth Chrome instance automatically
    browser = await uc.start()

    url = "https://genius.com/Paramore-whoa-lyrics"
    page = await browser.get(url)

    # Give the background Turnstile script 4-5 seconds to auto-solve
    await page.sleep(5) 

    # Extract the loaded page HTML
    html = await page.get_content()

    # Hand off to BeautifulSoup just like before
    soup = BeautifulSoup(html, "html.parser")
    lyric_containers = soup.find_all(attrs={"data-lyrics-container": "true"})

    lyrics = [c.get_text(separator="\n") for c in lyric_containers]
    print("\n".join(lyrics)[:500])

    await browser.stop()

await get_lyrics()


20 Contributors
Whoa Lyrics
The seventh song off Paramore’s debut album, “Whoa” talks about how we hurt ourselves emotionally when we feel with our heart or follow our heart rather than thinking with our head. Hayley has stated in a… 
Read More
 
[Verse 1]
I've hoped for change and it gets better every day
I've hoped for change, but still, I feel the same
[Pre-Chorus]
There's something wrong 'cause everybody knows
That we can do this on our own
[Chorus]
And we've got everybody singin'
Woah, woah


TypeError: object NoneType can't be used in 'await' expression

In [ ]:
# SKIP (ALSO diagnostic for debugging data on albums)

r = requests.get("https://genius.com/albums/Paramore/All-we-know-is-falling")
soup = BeautifulSoup(r.text, "html.parser")

song_links = soup.find_all("a", href=lambda v: v and v.endswith("-lyrics"))
for link in song_links:
    # Print the parent's class so we can see what container each link lives in
    print(repr(link.get_text(strip=True)), "→", link.parent.get("class"))






In [8]:
# 2026-11-06 Update: Alas, this no longer works, but we can take alternate measures.  
# STORING FOR ARCHIVAL MEMORY. 
# ebb: Here are two scraping functions that work as of 30 March 2026! 
# READ FROM THE BOTTOM TO TOP. 
# Supply your album title and define your relative path to your output directory where you want to store lyrics text files (one text file for each song).
# That information is needed to start up the get_lyrics_from_album_url(album_url, output_dir)
# And as that function pulls song titles from the album page, it then triggers the get_lyrics() function, sending it the song URL.
# get_lyrics(), the first function you see here, delivers the song lyrics back to the main function, which then writes the lyrics to output.
# Genius keeps changing its HTML page metadata, so we probably will have to keep updating this cell over time.
def get_lyrics(song_url):
    r = requests.get(song_url)
    soup = BeautifulSoup(r.text, "html.parser")
    
    lyric_divs = soup.find_all("div", attrs={"data-lyrics-container": "true"})
    
    lyrics = []
    for div in lyric_divs:
        for br in div.find_all("br"):
            br.replace_with("\n")
        text = div.get_text()
        
        if not text.strip():
            continue
        
        # If the div contains a bracket marker, trim everything before the first one
        first_bracket = text.find("[")
        if first_bracket != -1:
            text = text[first_bracket:]
            
        # ebb: The next two lines remove the square-bracketed stuff 
        # like `[Verse 1]`, `[Outro]`, etc. with the regular expression`\[.*?\]` 
        # If you want to KEEP these for use in tagging later for an XML project, 
        # comment out the next two lines. 
        text = re.sub(r'\[.*?\]', '', text)
        text = re.sub(r'\n{3,}', '\n\n', text).strip()
        lyrics.append(text)
    
    return "\n".join(lyrics).strip()
    
def get_lyrics_from_album_url(album_url, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    r = requests.get(album_url)
    soup = BeautifulSoup(r.text, "html.parser")
    
    # Only look for lyrics links inside chart_row-content containers
    track_rows = soup.find_all("div", class_="chart_row-content")
    
    corpus = {}
    for row in track_rows:
        link = row.find("a", href=lambda v: v and v.endswith("-lyrics"))
        if link:
            song_url = link["href"]
            song_title = link.get_text(strip=True)
            print(f"Fetching: {song_title}")
            lyrics = get_lyrics(song_url)
            # This runs the function above!!!
            corpus[song_title] = lyrics
            
            # Clean up (remove spaces, path separators, etc. from the song title) for use as a filename
            # Also remove the "Lyrics" from the end of the song title with a regex subsitutitons on Lyrics$ (at the end of the string), 
            # and removing punctuation from titles with a regex character class.
            safe_title = song_title.replace("/", "-").replace(" ", "_").strip()
            safe_title = re.sub(r'Lyrics$', '', safe_title).strip()
            safe_title = re.sub(r"[,.'!?;:]", '', safe_title)
            filepath = os.path.join(output_dir, f"{safe_title}.txt")
            
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(lyrics)
                
            print(f"  Saved: {filepath}")
            time.sleep(1)
    
    return corpus

# Usage
album_url = "https://genius.com/albums/Paramore/All-we-know-is-falling"
output_dir = "output/paramore-all-we-know-is-falling"
# album_url = "https://genius.com/albums/Paramore/Riot"
# output_dir="output/riot"
corpus = get_lyrics_from_album_url(album_url, output_dir)

In [6]:
import os
import re
import asyncio
import nodriver as uc
from bs4 import BeautifulSoup

async def get_lyrics(browser, song_url):
    """Fetches and cleans lyrics by opening and closing a dedicated tab."""
    # Force nodriver to open a fresh tab target specifically for this song
    page = await browser.get(song_url, new_tab=True)
    
    # Give Turnstile a few seconds to auto-solve in the new tab
    await page.sleep(4) 
    
    # Extract the fully rendered DOM content
    html = await page.get_content()
    soup = BeautifulSoup(html, "html.parser")
    
    # Safely destroy this specific tab target
    await page.close()
    
    lyric_divs = soup.find_all("div", attrs={"data-lyrics-container": "true"})
    
    lyrics = []
    for div in lyric_divs:
        for br in div.find_all("br"):
            br.replace_with("\n")
        text = div.get_text()
        
        if not text.strip():
            continue
        
        # Trim everything before the first structural bracket marker
        first_bracket = text.find("[")
        if first_bracket != -1:
            text = text[first_bracket:]
            
        # Strip or keep bracketed markers depending on your XML schema requirements
        text = re.sub(r'\[.*?\]', '', text)
        text = re.sub(r'\n{3,}', '\n\n', text).strip()
        lyrics.append(text)
    
    return "\n".join(lyrics).strip()
    
async def get_lyrics_from_album_url(album_url, output_dir):
    """Processes an entire album sequentially by handing off explicit tab objects."""
    os.makedirs(output_dir, exist_ok=True)
    
    # Start the base stealth browser profile
    browser = await uc.start()
    
    # Extract the main index page using the default base tab
    base_page = await browser.get(album_url)
    await base_page.sleep(4)
    html = await base_page.get_content()
    
    soup = BeautifulSoup(html, "html.parser")
    track_rows = soup.find_all("div", class_="chart_row-content")
    
    corpus = {}
    for row in track_rows:
        link = row.find("a", href=lambda v: v and v.endswith("-lyrics"))
        if link:
            song_url = link["href"]
            song_title = link.get_text(strip=True)
            print(f"Fetching: {song_title}")
            
            try:
                # Run our isolated tab process
                lyrics = await get_lyrics(browser, song_url)
                corpus[song_title] = lyrics
                
                # Clean up punctuation and strings for valid local file names
                safe_title = song_title.replace("/", "-").replace(" ", "_").strip()
                safe_title = re.sub(r'Lyrics$', '', safe_title).strip()
                safe_title = re.sub(r"[,.'!?;:]", '', safe_title)
                filepath = os.path.join(output_dir, f"{safe_title}.txt")
                
                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(lyrics)
                    
                print(f"  Saved: {filepath}")
                
            except Exception as e:
                print(f"  Failed to scrape {song_title}: {e}")
                
            # Maintain a polite structural pause between tabs
            await asyncio.sleep(2)
            
    # Terminate the core binary process once the loop finishes
    browser.stop()
    return corpus

# --- Run the pipeline in your Jupyter notebook cell ---
album_url = "https://genius.com/albums/Paramore/All-we-know-is-falling"
output_dir = "output-2026-11/paramore-all-we-know-is-falling"

corpus = await get_lyrics_from_album_url(album_url, output_dir)


Fetching: All We KnowLyrics
  Saved: output-2026-11/paramore-all-we-know-is-falling/All_We_Know.txt
Fetching: PressureLyrics
  Saved: output-2026-11/paramore-all-we-know-is-falling/Pressure.txt
Fetching: EmergencyLyrics
  Saved: output-2026-11/paramore-all-we-know-is-falling/Emergency.txt
Fetching: BrighterLyrics
  Saved: output-2026-11/paramore-all-we-know-is-falling/Brighter.txt
Fetching: Here We Go AgainLyrics
  Saved: output-2026-11/paramore-all-we-know-is-falling/Here_We_Go_Again.txt
Fetching: Never Let This GoLyrics
  Saved: output-2026-11/paramore-all-we-know-is-falling/Never_Let_This_Go.txt
Fetching: WhoaLyrics
  Saved: output-2026-11/paramore-all-we-know-is-falling/Whoa.txt
Fetching: ConspiracyLyrics
  Saved: output-2026-11/paramore-all-we-know-is-falling/Conspiracy.txt
Fetching: FranklinLyrics
  Saved: output-2026-11/paramore-all-we-know-is-falling/Franklin.txt
Fetching: My HeartLyrics
  Saved: output-2026-11/paramore-all-we-know-is-falling/My_Heart.txt


## Next step: Use the Genius API to lookup albums
(2026-11-06)
Taking this further, we can use the Genius API here to pull the album URLs for us, and then retrieve the lyrics from their URLs.


In [ ]:
import os
import re
import asyncio
import requests
from dotenv import load_dotenv
import nodriver as uc
from bs4 import BeautifulSoup

# Load API credentials
load_dotenv()
GENIUS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN")

def discover_album_urls_via_api(artist_name, album_name):
    """Step 1: Uses your token to find the album and fetch clean track URLs."""
    headers = {"Authorization": f"Bearer {GENIUS_TOKEN}"}
    
    # 1. Search for a track to locate the album object
    search_url = "https://genius.com"
    query = f"{artist_name} {album_name}"
    response = requests.get(search_url, headers=headers, params={"q": query})
    
    if response.status_code != 200 or not response.json()["response"]["hits"]:
        print(f" Could not find an API match for '{query}'.")
        return []
        
    # Grab the album details from the first matching track hit
    first_hit = response.json()["response"]["hits"][0]["result"]
    if not first_hit.get("album"):
        print(" Found songs, but couldn't resolve the primary album container structure.")
        return []
        
    album_id = first_hit["album"]["id"]
    album_title = first_hit["album"]["name"]
    print(f" Found Official API Match: '{album_title}' (ID: {album_id})")
    
    # 2. Fetch the complete, structured tracklist metadata
    tracks_url = f"https://genius.com{album_id}/tracks"
    tracks_response = requests.get(tracks_url, headers=headers)
    tracks_data = tracks_response.json()["response"]["tracks"]
    
    track_list = []
    for item in tracks_data:
        song_node = item["song"]
        track_list.append({
            "title": song_node["title"],
            "url": song_node["url"]
        })
    return track_list

async def extract_lyrics_from_url(browser, song_url):
    """Step 2: Opens a dedicated stealth tab to pull text behind Turnstile."""
    page = await browser.get(song_url, new_tab=True)
    await page.sleep(4) # Let Turnstile clear
    
    html = await page.get_content()
    await page.close()
    
    soup = BeautifulSoup(html, "html.parser")
    lyric_divs = soup.find_all("div", attrs={"data-lyrics-container": "true"})
    
    lyrics = []
    for div in lyric_divs:
        for br in div.find_all("br"):
            br.replace_with("\n")
        text = div.get_text()
        
        if not text.strip():
            continue
            
        first_bracket = text.find("[")
        if first_bracket != -1:
            text = text[first_bracket:]
            
        text = re.sub(r'\[.*?\]', '', text)
        text = re.sub(r'\n{3,}', '\n\n', text).strip()
        lyrics.append(text)
        
    return "\n".join(lyrics).strip()

async def scrape_album_by_name(artist_name, album_name, output_base_dir="output"):
    """Step 3: Manages the entire pipeline from text search to local text files."""
    # 1. Discover tracks via API
    tracks = discover_album_urls_via_api(artist_name, album_name)
    if not tracks:
        return {}
        
    # Set up clean local paths
    safe_album_name = album_name.lower().replace(" ", "_")
    output_dir = os.path.join(output_base_dir, f"{artist_name.lower()}-{safe_album_name}")
    os.makedirs(output_dir, exist_ok=True)
    
    # 2. Launch our Turnstile-immune browser instance
    print(f"Launching stealth browser to process {len(tracks)} tracks...")
    browser = await uc.start()
    corpus = {}
    
    for track in tracks:
        song_title = track["title"]
        song_url = track["url"]
        print(f" Scraping: {song_title}")
        
        try:
            lyrics = await extract_lyrics_from_url(browser, song_url)
            corpus[song_title] = lyrics
            
            # Format title safely for standard local file systems
            safe_title = song_title.replace("/", "-").replace(" ", "_").strip()
            safe_title = re.sub(r"[,.'!?;:]", '', safe_title)
            filepath = os.path.join(output_dir, f"{safe_title}.txt")
            
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(lyrics)
            print(f"   Saved: {filepath}")
            
        except Exception as e:
            print(f"   Failed to pull {song_title}: {e}")
            
        await asyncio.sleep(2) # Behavioral delay
        
    browser.stop()
    print("\n Album data sync completed successfully!")
    return corpus

# --- Native Jupyter Execution ---
# corpus = await scrape_album_by_name("Paramore", "Riot")


In [ ]:
# Inputting album info to launch the function

albums_to_get = [
    ("Paramore", "All We Know Is Falling"),
    ("Paramore", "Riot"),
    ("Paramore", "Brand New Eyes")
]

for artist, album in albums_to_get:
    await scrape_album_by_name(artist, album)
